# Étude Complète du Retour sur Investissement (ROI) Marketing

Ce notebook présente une étude complète de bout en bout pour analyser et modéliser l'impact des budgets publicitaires (TV, Radio, Réseaux Sociaux) sur les Ventes (`Sales`).

### Objectifs :
1. **Analyse Exploratoire des Données (EDA) :** Visualiser les distributions et les équilibres des variables.
2. **Nettoyage des Données :** Traiter les valeurs manquantes et éliminer les valeurs aberrantes (outliers).
3. **Analyse de Corrélation & Multicolinéarité :** Étudier les corrélations et calculer le VIF (Variance Inflation Factor).
4. **Sélection de Caractéristiques (Feature Selection) :** Déterminer les variables clés via RFE.
5. **Modélisation Machine Learning :** Entraîner et comparer différents modèles de régression pour prédire les ventes.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Configuration graphique
%matplotlib inline
sns.set_theme(style="whitegrid")

## 1. Analyse Exploratoire des Données (EDA)

Nous commençons par charger les données brutes pour comprendre les types de variables et repérer les premières anomalies.

In [ ]:
# Chargement du fichier brut
df_raw = pd.read_csv('data/Dummy Data HSS.csv')
print(f"Taille du jeu de données brut : {df_raw.shape}")
df_raw.head()

In [ ]:
# Informations générales
df_raw.info()

In [ ]:
# Statistiques descriptives de base
df_raw.describe(include='all').T

In [ ]:
# Vérification de l'équilibre des types d'influenceurs
inf_counts = df_raw['Influencer'].value_counts()
inf_pct = df_raw['Influencer'].value_counts(normalize=True) * 100
for idx in inf_counts.index:
    print(f"{idx}: {inf_counts[idx]} ({inf_pct[idx]:.2f}%)")

In [ ]:
# Tracé des histogrammes des variables numériques et countplot pour Influencer
fig, axes = plt.subplots(3, 2, figsize=(15, 18))
fig.suptitle("Distribution des Variables du Jeu de Données Brut", fontsize=16, fontweight='bold', y=0.96)

sns.histplot(df_raw['TV'].dropna(), kde=True, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Distribution du Budget TV', fontsize=12, fontweight='bold')

sns.histplot(df_raw['Radio'].dropna(), kde=True, ax=axes[0, 1], color='lightcoral')
axes[0, 1].set_title('Distribution du Budget Radio', fontsize=12, fontweight='bold')

sns.histplot(df_raw['Social Media'].dropna(), kde=True, ax=axes[1, 0], color='lightgreen')
axes[1, 0].set_title('Distribution du Budget Réseaux Sociaux', fontsize=12, fontweight='bold')

sns.histplot(df_raw['Sales'].dropna(), kde=True, ax=axes[1, 1], color='gold')
axes[1, 1].set_title('Distribution des Ventes (Target)', fontsize=12, fontweight='bold')

sns.countplot(x='Influencer', data=df_raw, ax=axes[2, 0], order=['Mega', 'Macro', 'Micro', 'Nano'], palette='viridis')
axes[2, 0].set_title("Répartition des types d'influenceurs", fontsize=12, fontweight='bold')

axes[2, 1].axis('off')
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

## 2. Nettoyage des Données

Nous procédons aux étapes de nettoyage suivantes :
1. **Suppression de la variable `Influencer`** (qui ne fait pas partie de l'étude demandée).
2. **Suppression des valeurs manquantes (NaN).**
3. **Suppression des outliers (valeurs extrêmes)** à l'aide de la méthode de l'écart interquartile (IQR).

In [ ]:
# Suppression de la colonne Influencer
df_clean = df_raw.drop(columns=['Influencer'])

# Analyse et suppression des valeurs manquantes
print("Valeurs manquantes par colonne avant nettoyage :")
print(df_clean.isnull().sum())

df_clean = df_clean.dropna()
print(f"Taille après suppression des NaNs : {df_clean.shape}")

In [ ]:
# Élimination des outliers par la méthode IQR
numeric_cols = ['TV', 'Radio', 'Social Media', 'Sales']
outlier_mask = pd.Series(False, index=df_clean.index)

for col in numeric_cols:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    col_outliers = (df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)
    print(f"Outliers détectés dans {col} : {col_outliers.sum()} (bornes : [{lower_bound:.2f}, {upper_bound:.2f}])")
    outlier_mask = outlier_mask | col_outliers

df_clean = df_clean[~outlier_mask]
print(f"\nTaille finale du jeu de données nettoyé : {df_clean.shape}")

# Enregistrement des données propres
df_clean.to_csv('data/Clean_Data_HSS.csv', index=False)

## 3. Analyse de Corrélation et Sélection des Caractéristiques

In [ ]:
# Matrice de corrélation
plt.figure(figsize=(8, 6))
sns.heatmap(df_clean.corr(), annot=True, cmap='coolwarm', fmt=".4f", linewidths=0.5, vmin=-1, vmax=1)
plt.title("Matrice de Corrélation des Données Nettoyées", fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# Facteur d'Inflation de la Variance (VIF) pour mesurer la colinéarité
X_vars = df_clean[['TV', 'Radio', 'Social Media']]
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vars.columns
vif_data["VIF"] = [variance_inflation_factor(X_vars.values, i) for i in range(len(X_vars.columns))]
vif_data

In [ ]:
# Feature Selection avec RFE (Élimination Récursive de Caractéristiques)
X = df_clean[['TV', 'Radio', 'Social Media']]
y = df_clean['Sales']

model = LinearRegression()
for n in range(1, 4):
    rfe = RFE(estimator=model, n_features_to_select=n)
    rfe.fit(X, y)
    print(f"Top {n} variable(s) sélectionnée(s) : {X.columns[rfe.support_].tolist()}")

## 4. Modélisation et Comparaison de Modèles

Nous divisons les données nettoyées en ensembles d'entraînement (80 %) et de test (20 %). Nous testons plusieurs approches :
1. **Régression Linéaire Simple** (uniquement la `TV` car c'est la variable dominante).
2. **Régression Linéaire Multiple** (avec les 3 variables explicatives).
3. **Régression Ridge** (pénalité L2 pour limiter les effets de la multicolinéarité).
4. **Régression Lasso** (pénalité L1 pour la sélection automatique de variables).
5. **Random Forest** (modèle non linéaire basé sur des arbres de décision).

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

results = {}

def run_experiment(name, model, X_tr, X_te):
    model.fit(X_tr, y_train)
    preds = model.predict(X_te)
    
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    
    results[name] = {
        'R2 Score': r2,
        'RMSE': rmse,
        'MAE': mae
    }
    return preds

In [ ]:
# 1. Simple LR (TV uniquement)
preds_slr = run_experiment("Simple LR (TV only)", LinearRegression(), X_train[['TV']], X_test[['TV']])

# 2. Multiple LR
preds_mlr = run_experiment("Multiple LR (All features)", LinearRegression(), X_train, X_test)

# 3. Ridge
preds_ridge = run_experiment("Ridge Regression", RidgeCV(alphas=np.logspace(-4, 4, 100)), X_train, X_test)

# 4. Lasso
preds_lasso = run_experiment("Lasso Regression", LassoCV(alphas=np.logspace(-4, 4, 100), cv=5, random_state=42), X_train, X_test)

# 5. Random Forest
preds_rf = run_experiment("Random Forest", RandomForestRegressor(n_estimators=100, random_state=42), X_train, X_test)

In [ ]:
# Affichage des résultats
results_df = pd.DataFrame(results).T.sort_values(by='R2 Score', ascending=False)
results_df

In [ ]:
# Visualisation : Valeurs Réelles vs Valeurs Prédites
plt.figure(figsize=(10, 8))
plt.scatter(y_test, preds_mlr, alpha=0.6, color='blue', label='Multiple LR')
plt.scatter(y_test, preds_rf, alpha=0.4, color='green', marker='x', label='Random Forest')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
plt.title("Valeurs Réelles vs Valeurs Prédites (Sales)", fontsize=14, fontweight='bold')
plt.xlabel("Valeurs Réelles (Sales)")
plt.ylabel("Prédictions")
plt.legend()
plt.show()

## 5. Conclusions et Recommandations

1. **Le rôle clé de la TV :** Le budget TV explique à lui seul plus de 99.9% de la variance des ventes. C'est l'investissement le plus rentable et le plus direct.
2. **Le piège de la colinéarité :** L'inclusion de la variable `Radio` dans une régression linéaire multiple standard introduit de la multicolinéarité (VIF $\approx 12$), ce qui provoque un coefficient négatif aberrant pour la `Radio`. Le modèle Lasso a d'ailleurs choisi d'annuler complètement les coefficients de `Radio` et `Social Media`.
3. **Recommandation finale :** Pour des décisions stratégiques simples, un modèle linéaire basé uniquement sur la `TV` est optimal. Si l'on souhaite combiner tous les canaux, il faut privilégier la régression **Lasso** ou utiliser des algorithmes d'ensemble (comme **Random Forest** ou **XGBoost**).